In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

# Load image
img = cv2.imread("panel.jpeg")

# Resize (optional)
img = cv2.resize(img, (800,600))

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Noise removal
blur = cv2.GaussianBlur(gray, (5,5), 0)

# ================= EDGE DETECTION (CRACKS) =================
edges = cv2.Canny(blur, 50, 150)

# ================= SPOT DETECTION (DUST / STAINS) ==========
params = cv2.SimpleBlobDetector_Params()
params.filterByArea = True
params.minArea = 100
params.maxArea = 5000

detector = cv2.SimpleBlobDetector_create(params)
keypoints = detector.detect(blur)

spot_img = cv2.drawKeypoints(img, keypoints, None,
                             (0,0,255),
                             cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

# ================= DEFECT AREA CALCULATION =================
_, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)

total_pixels = img.shape[0] * img.shape[1]
defect_pixels = cv2.countNonZero(thresh)

defect_percentage = (defect_pixels / total_pixels) * 100

# ================= SEVERITY CLASSIFICATION =================
if defect_percentage < 2:
    severity = "LOW"
elif defect_percentage < 6:
    severity = "MEDIUM"
else:
    severity = "HIGH"

# ================= DISPLAY RESULTS =================
print("Defect Area Percentage:", round(defect_percentage,2), "%")
print("Severity Level:", severity)

cv2.putText(spot_img, f"Defect: {round(defect_percentage,2)}%",
            (20,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

cv2.putText(spot_img, f"Severity: {severity}",
            (20,80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

cv2_imshow(img)
cv2_imshow(edges)
cv2_imshow(spot_img)

cv2.waitKey(0)
cv2.destroyAllWindows()